# 国会图书馆 SRU 抓取结果 · 数据浏览与描述统计

本笔记本读取 **`01周/data/lc_sru_export/`** 里由前一实验导出的数据：

- **`书目摘要.parquet`**：6 列速览（便于和 Excel 对齐）。
- **`marc_flat.parquet`**（重新跑过 `loc_sru_lab` §5 后生成）：**MARC 全字段长表**，每条书目多行，含 **650、651** 等所有子字段；做主题统计应优先用此文件。

你将：

1. 看清各表结构与示例行。
2. 选定一本书：摘要行 + 可选从 **`marc_flat`** 按 `bib_id` 查看全部字段。
3. 描述统计：出版年、出版社；以及用 **`marc_flat`** 统计 **650** 等字段。

（若需要 **一书一行** 的 `marc_bibli_one_row.parquet`，请在 `loc_sru_lab.ipynb` 中于 §5 分页导出完成后运行 **§5b**。）



## 1. 路径与读取 Parquet

以下路径相对于 **`上机实验/01周`**（请先在此处打开 Jupyter / 或保证工作目录为本周文件夹）。


In [ ]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET

import pandas as pd

# 与抓数实验输出位置一致
BASE = Path.cwd()
DATA_DIR = BASE / "data" / "lc_sru_export"
PARQUET_PATH = DATA_DIR / "书目摘要.parquet"
CSV_PATH = DATA_DIR / "书目摘要.csv"
XML_DIR = DATA_DIR / "xml_pages"

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f"未找到 {PARQUET_PATH.resolve()}\n请先运行 loc_sru_lab.ipynb 的 §5 完成抓取与导出。"
    )

df = pd.read_parquet(PARQUET_PATH)
print("Parquet 路径:", PARQUET_PATH.resolve())
print("行数 × 列数:", df.shape)
print("列名:", list(df.columns))
print()
df.info()
print()
print("前 3 行预览：")
df.head(3)


## 1b. MARC 全字段长表 `marc_flat.parquet`

长表列含义：`bib_id`（001 记录号）、`source_xml`（来自哪一页 XML）、`field_order`（本书内顺序）、`kind`（leader / control / data）、`tag`、`ind1`、`ind2`、`subfield_code`、`value`。

**一行 = leader 整段，或一个 controlfield，或 datafield 里的一个 subfield。** 用 `bib_id` 分组即可还原整条 MARC。


In [ ]:
FLAT_PATH = DATA_DIR / "marc_flat.parquet"
if FLAT_PATH.exists():
    df_flat = pd.read_parquet(FLAT_PATH)
    print("marc_flat 路径:", FLAT_PATH.resolve())
    print("行数 × 列数:", df_flat.shape)
    print(df_flat.head(8))
    n650 = df_flat[(df_flat["kind"] == "data") & (df_flat["tag"] == "650")].shape[0]
    print("\n650 子字段行数（全库）:", n650)
else:
    df_flat = None
    print("未找到 marc_flat.parquet — 请在 loc_sru_lab.ipynb 中重新运行 §5 导出（新版会生成该文件）。")


## 2. 表结构速览（书目摘要）

下面看 **6 列摘要表** 的非空率等。**完整 MARC** 请用上一节的 **`df_flat`**（若已生成）或仍可到 XML 中核对。


In [ ]:
# 非空率、唯一值数量（粗看数据质量）
summary = pd.DataFrame({
    "非空条数": df.notna().sum(),
    "唯一值数": df.nunique(dropna=True),
})
summary


## 3. 选定一本书：看「摘要行」在说什么

从表中任取一行（这里用 **第 1 行**；你可以改成 `df.loc[df["001"] == "某记录号"]`）。各列对应 MARC 里常抽的：记录号、索书号、正题名、出版地、出版者、出版年项。


In [ ]:
# 改 index 可换别的行，或按 001 筛选
ROW_I = 0
one = df.iloc[ROW_I]
print("（行号）", ROW_I, "  记录号 001 =", one["001"])
one.to_frame("值")


## 4. 在分页 XML 里找回这本书的 **完整 MARC**

`书目摘要` 只存了 6 列。要「看里面具体内容」，需要到 `xml_pages/page_*.xml` 里找 **`<controlfield tag="001">` 与所选记录号相同** 的那条 `<record>`，下面把各 `datafield` 打成易读列表（仍比馆员用的编目屏幕粗糙，但比 6 列全）。


In [ ]:
MARC_URI = "http://www.loc.gov/MARC21/slim"
NS = {"m": MARC_URI}


def find_marc_record(record_id: str, xml_dir: Path):
    """在 xml_pages 中扫描，返回 (文件路径, record 元素) 或 (None, None)"""
    record_id = str(record_id).strip()
    for path in sorted(xml_dir.glob("page_*.xml")):
        root = ET.parse(path).getroot()
        for rec in root.findall(".//{%s}record" % MARC_URI):
            for cf in rec.findall("m:controlfield", NS):
                if cf.get("tag") == "001" and (cf.text or "").strip() == record_id:
                    return path, rec
    return None, None


def marc_record_to_lines(rec) -> list[str]:
    """把一条 MARC 列成多行：标签 + 子字段"""
    lines: list[str] = []
    for cf in rec.findall("m:controlfield", NS):
        tag = cf.get("tag", "")
        lines.append(f"  {tag:3}  { (cf.text or '').strip() }")
    for df in rec.findall("m:datafield", NS):
        tag = df.get("tag", "")
        ind = (df.get("ind1", " ") or " ") + (df.get("ind2", " ") or " ")
        parts = []
        for sf in df.findall("m:subfield", NS):
            code = sf.get("code", "")
            parts.append(f"${code} { (sf.text or '').strip() }")
        lines.append(f"  {tag:3} {ind}  {' | '.join(parts)}")
    return lines


target_id = str(one["001"])
src_path, marc_rec = find_marc_record(target_id, XML_DIR)
if marc_rec is None:
    print("未在", XML_DIR, "中找到 001 =", target_id, "（可检查是否只导出了部分页）")
else:
    print("来源文件:", src_path.name)
    print("--- MARC 展开（节选前 40 行，避免刷屏）---")
    all_lines = marc_record_to_lines(marc_rec)
    for line in all_lines[:40]:
        print(line)
    if len(all_lines) > 40:
        print("  ... 共", len(all_lines), "行，其余略")


## 5. 描述统计

在**仅有摘要列**的前提下，可做：

- **出版年**：从 `260/264 $c` 用正则抓 4 位年（会漏印「约 19--」等，属粗统计）。
- **出版者**：`260/264 $b` 频数前名。
- **索书号大类**：取 `050` 第一截（如 `DS793`），仅作结构感演示。


In [ ]:
col_year = "260/264 $c"
col_pub = "260/264 $b"
col_cls = "050"


def extract_year(s):
    if not isinstance(s, str) or not s.strip():
        return pd.NA
    m = re.search(r"(17|18|19|20)\d{2}", s)
    return int(m.group(0)) if m else pd.NA


df["_year"] = df[col_year].apply(extract_year)
year_counts = df["_year"].value_counts().sort_index()
print("可解析出版年（频数，按年）：")
display(year_counts.tail(15))  # Jupyter 里美观；纯终端可 print(year_counts)

print("\n出版者（前 15）：")
print(df[col_pub].value_counts().head(15))

print("\n050 索书号前缀（取空格前第一段，前 15）：")
prefix = df[col_cls].astype(str).str.split().str[0]
print(prefix.value_counts().head(15))

print("\n年份缺失条数:", df["_year"].isna().sum(), "/", len(df))


## 6.（可选）换成 CSV

若环境读不了 Parquet，可改用：`df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")`。
